<a href="https://colab.research.google.com/github/uncertain-uncertainity/Learning/blob/main/chatbot_using_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YouTube Chatbot using LangChain | RAG Pipeline

Chat with any YouTube video in real time. Paste a video ID, and ask things like *"Is the topic of aliens discussed in this video?"* or *"Summarize this video in 5 bullet points"* instead of watching a 2-3 hour podcast end to end.

**Stack:** `YouTubeTranscriptApi` · `RecursiveCharacterTextSplitter` · Google Gemini embeddings (`GoogleGenerativeAIEmbeddings`) · FAISS vector store · `PromptTemplate` · Gemini 2.5 Flash (`ChatGoogleGenerativeAI`)

### RAG pipeline overview
| Stage | What we do |
|---|---|
| **1. Indexing** | Load the transcript → split into chunks → embed → store in FAISS |
| **2. Retrieval** | Embed the user's query and fetch the most similar chunks |
| **3. Augmentation** | Merge retrieved chunks + question into one prompt |
| **4. Generation** | Send the prompt to the LLM and get a grounded answer |

At the end, we wire everything into a single LangChain **chain** so one `.invoke()` call runs the whole pipeline.

## Setup: Gemini API key

Get a free key from [Google AI Studio](https://aistudio.google.com/apikey).

**Do not paste your key directly into the notebook.** Store it in Colab's *Secrets* panel (the 🔑 icon in the left sidebar) under the name `GOOGLE_API_KEY` and enable notebook access. If the secret isn't found, the cell below falls back to a hidden prompt.

In [1]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    # Not on Colab, or the secret isn't set -> ask for the key without echoing it
    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


## Install libraries

In [2]:
!pip install -q "youtube-transcript-api>=1.0.0" langchain-community langchain-core \
               langchain-text-splitters langchain-google-genai faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth

## Imports

In [3]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_871/1315767256.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Step 1a - Indexing (Document Ingestion)

We use the **YouTube Transcript API** to pull the transcript of a video directly (LangChain's built-in `YoutubeLoader` was flaky for some videos, so calling YouTube's transcript API ourselves is more reliable).

- Pass **only the video ID**, not the full URL. For `https://www.youtube.com/watch?v=Gfr50f6ZBvo` the ID is `Gfr50f6ZBvo`.
- Set `languages` to the language of the captions. Hindi videos usually have **no English transcript**, so switch `["en"]` to `["hi"]` for them.

In [4]:
video_id = "Gfr50f6ZBvo"   # only the ID, not the full URL
languages = ["en"]         # use ["hi"] for Hindi videos (or ["hi", "en"] to try both in order)

try:
    ytt_api = YouTubeTranscriptApi()
    fetched = ytt_api.fetch(video_id, languages=languages)

    # List of dicts: {"text": ..., "start": ..., "duration": ...}  (one entry per subtitle line)
    transcript_list = fetched.to_raw_data()

    # Flatten the broken-up subtitle lines into one long string
    transcript = " ".join(chunk["text"] for chunk in transcript_list)

    print(f"Fetched {len(transcript_list)} subtitle lines, {len(transcript):,} characters")
    print(transcript[:500], "...")

except TranscriptsDisabled:
    print("No captions available for this video.")
except NoTranscriptFound:
    print(f"No transcript found for languages {languages}. Try another language code, e.g. ['hi'].")

Fetched 3790 subtitle lines, 133,836 characters
the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful  ...


The raw transcript is a list of dictionaries. Each entry has the subtitle `text`, the `start` time (seconds) it appears on screen, and its `duration`.

In [5]:
transcript_list[:5]

[{'text': 'the following is a conversation with',
  'start': 0.08,
  'duration': 3.44},
 {'text': 'demus hasabis', 'start': 1.76, 'duration': 4.96},
 {'text': 'ceo and co-founder of deepmind', 'start': 3.52, 'duration': 5.119},
 {'text': 'a company that has published and builds',
  'start': 6.72,
  'duration': 4.48},
 {'text': 'some of the most incredible artificial',
  'start': 8.639,
  'duration': 4.561}]

## Step 1b - Indexing (Text Splitting)

A 2-hour transcript is far too long to send to an LLM in one go, so we split it into small overlapping chunks using `RecursiveCharacterTextSplitter`. The overlap keeps context from being lost at chunk boundaries. Feel free to experiment with different `chunk_size` / `chunk_overlap` values.

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [7]:
len(chunks)

168

In [8]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

Each chunk is converted into a high-dimensional vector with **Google Gemini embeddings**, and the vectors are stored in a **FAISS** vector store for fast similarity search.

> If you hit rate-limit (429) errors on the Gemini free tier while embedding all chunks, wait a minute and re-run, or use a larger `chunk_size` to reduce the number of chunks.

In [10]:
import time

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

BATCH_SIZE = 90      # stay under the free-tier limit of 100 requests/minute
WAIT_SECONDS = 60

vector_store = None
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    while True:
        try:
            if vector_store is None:
                vector_store = FAISS.from_documents(batch, embeddings)
            else:
                vector_store.add_documents(batch)
            break
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print(f"Rate limited, waiting {WAIT_SECONDS}s and retrying...")
                time.sleep(WAIT_SECONDS)
            else:
                raise
    print(f"Indexed {min(i + BATCH_SIZE, len(chunks))}/{len(chunks)} chunks")
    if i + BATCH_SIZE < len(chunks):
        time.sleep(WAIT_SECONDS)

Indexed 90/168 chunks
Indexed 168/168 chunks


In [11]:
vector_store.index_to_docstore_id

{0: '97983983-e57b-40d8-8e9b-4d4df27fd90e',
 1: 'b7078134-780d-4335-8b54-088e8ca56162',
 2: '4f378ae3-ab25-431a-9409-e580f7d2e026',
 3: '74c24a81-35de-4966-bc9d-9fb9f4977fe0',
 4: 'ab20bff3-1cf3-452e-89db-0d7d61e06dd5',
 5: '4c9b384a-1df9-4a33-915e-0f895068478c',
 6: 'ca875912-9fec-48b9-a310-8567422813f0',
 7: 'a8631770-7235-47f6-80e1-fbeb0ae22285',
 8: '0db04c41-3379-4445-ad62-b60256404e44',
 9: '7d6e006f-9aee-4096-acbe-a2693a71674c',
 10: 'fdcd3eb8-b249-4f40-86ca-54f514d5bb35',
 11: '01cb4fc8-c691-4c3b-919f-a60ece640690',
 12: '97fb7201-2914-4dd0-ad7f-af9091d41211',
 13: '8aeaee1a-4ad0-4663-bf94-194a6ad7567b',
 14: '43ee4f4d-4efa-42c2-a77e-c9e1450652c4',
 15: '74c7ba94-3f9f-4314-b313-c49f145551b2',
 16: 'e051bb24-7a72-44bd-b161-92a989ce901c',
 17: '5d47fe66-6fea-4ca1-a26c-5ba93d479a15',
 18: '0c031322-cacc-4f31-948a-33246891e958',
 19: '6c6c83c4-dbf4-4307-b1f2-5595f0df9697',
 20: 'da9bdb44-d038-4890-825b-7e447be380ce',
 21: 'fbbdd4b5-7091-4edc-9db4-52efd513c042',
 22: '7188ebd2-7394-

Every chunk now has an ID in the vector store. Let's look up the last chunk by its ID.

In [ ]:
last_chunk_id = vector_store.index_to_docstore_id[len(chunks) - 1]
vector_store.get_by_ids([last_chunk_id])

## Step 2 - Retrieval

A **retriever** takes a query as input and returns a list of relevant documents. Internally it embeds the query, runs a similarity search over the vector store, and returns the closest chunks. Here we use plain similarity search and return the top 4 chunks.

In [12]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [13]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x79c29ac30ad0>, search_kwargs={'k': 4})

In [14]:
retriever.invoke('What is deepmind')

[Document(id='b58f74e5-db71-42c1-b90f-00d1b82923a3', metadata={}, page_content='i used to discuss um uh uh what were the sort of founding tenets of deep mind and it was very various things one was um algorithmic advances so deep learning you know jeff hinton and cohen just had just sort of invented that in academia but no one in industry knew about it uh we love reinforcement learning we thought that could be scaled up but also understanding about the human brain had advanced um quite a lot uh in the decade prior with fmri machines and other things so we could get some good hints about architectures and algorithms and and sort of um representations maybe that the brain uses so as at a systems level not at a implementation level um and then the other big things were compute and gpus right so we could see a compute was going to be really useful and it got to a place where it became commoditized mostly through the games industry and and that could be taken advantage of and then the final 

## Step 3 - Augmentation

We merge the retrieved chunks and the user's question into a single prompt. The prompt explicitly tells the model to answer **only from the transcript context** (answer grounding), which reduces hallucination.

In [23]:
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.2)

In [24]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables=["context", "question"]
)

In [25]:
question       = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [26]:
retrieved_docs

[Document(id='995b51be-2259-44c2-a785-794dffc91520', metadata={}, page_content="like room temperature superconductors or something on my list one day i'd like to like you know have an ai system to help build better optimized batteries all of these sort of mechanical things mr i think a systematic sort of search could be uh guided by a model could be um could be extremely powerful so speaking of which you have a paper on nuclear fusion uh magnetic control of tokamak plasmas to deep reinforcement learning so you uh you're seeking to solve nuclear fusion with deep rl so it's doing control of high temperature plasmas can you explain this work and uh can ai eventually solve nuclear fusion it's been very fun last year or two and very productive because we've been taking off a lot of my dream projects if you like of things that i've collected over the years of areas of science that i would like to i think could be very transformative if we helped accelerate and uh really interesting problems 

The retriever returns a *list* of documents, but a prompt needs a single string, so we concatenate the `page_content` of every retrieved chunk.

In [27]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"like room temperature superconductors or something on my list one day i'd like to like you know have an ai system to help build better optimized batteries all of these sort of mechanical things mr i think a systematic sort of search could be uh guided by a model could be um could be extremely powerful so speaking of which you have a paper on nuclear fusion uh magnetic control of tokamak plasmas to deep reinforcement learning so you uh you're seeking to solve nuclear fusion with deep rl so it's doing control of high temperature plasmas can you explain this work and uh can ai eventually solve nuclear fusion it's been very fun last year or two and very productive because we've been taking off a lot of my dream projects if you like of things that i've collected over the years of areas of science that i would like to i think could be very transformative if we helped accelerate and uh really interesting problems scientific challenges in of themselves this is energy so energy yes exactly so\

In [28]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [29]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      like room temperature superconductors or something on my list one day i'd like to like you know have an ai system to help build better optimized batteries all of these sort of mechanical things mr i think a systematic sort of search could be uh guided by a model could be um could be extremely powerful so speaking of which you have a paper on nuclear fusion uh magnetic control of tokamak plasmas to deep reinforcement learning so you uh you're seeking to solve nuclear fusion with deep rl so it's doing control of high temperature plasmas can you explain this work and uh can ai eventually solve nuclear fusion it's been very fun last year or two and very productive because we've been taking off a lot of my dream projects if you like of things that i've collected over the years of areas of science that i

## Step 4 - Generation

In [30]:
answer = llm.invoke(final_prompt)
print(answer.content)

[{'type': 'text', 'text': 'Yes, the topic of nuclear fusion is discussed. \n\nBased on the transcript, the discussion includes the following details:\n* **The Paper & Problem:** They discussed work published in a *Nature* paper on using deep reinforcement learning for the "magnetic control of tokamak plasmas."\n* **Why AI/Reinforcement Learning was used:** Plasma reaches extreme temperatures (around a million degrees Celsius, hotter than the sun), so no physical material can contain it; it must be held using powerful superconducting magnetic fields. Because plasma is unstable (like holding a mini star in a reactor), magnetic fields must be adjusted within milliseconds to contain what it does next. Traditional handcrafted, hard-coded controllers could not react in the moment, making reinforcement learning and prediction an ideal solution.\n* **The Achievements:** Using an AI controller and a plasma simulator, they were able to contain and hold the plasma into specific shapes (described 

## Building a Chain

So far we invoked the retriever, the prompt, and the LLM **manually, one by one**. Let's wire them into a chain so a single `invoke()` runs the whole pipeline and each component's output automatically becomes the next one's input.

The final chain is made of two parts:
1. A **parallel chain** that produces the prompt's two inputs at the same time:
   - `context`: question → retriever → `format_docs` (list of docs → one string)
   - `question`: question passed through unchanged
2. A **linear chain**: `prompt → llm → parser`

In [31]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [32]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

`format_docs` is a plain function, so we wrap it in `RunnableLambda` to make it a chain component.

In [33]:
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

In [34]:
parallel_chain.invoke('who is Demis')

{'context': "scenarios and try and improve your weaknesses but also find out what your unique skills and strengths are and then hone those so then that's what will be your super value in the world later on and if you can then combine those two things and find passions that you're genuinely excited about that intersect with what your unique strong skills are then you're you know you're on to something incredible and and you know i think you can make a huge difference in the world so let me ask about know yourself this is fun this is fun quick questions about day in the life the perfect day the perfect productive day in the life of demise's house yeah maybe uh maybe these days you're um there's a lot involved yeah it may be a slightly younger you could focus on a demonstration project maybe um how early do you wake up are you night owl do you wake up early in the morning what are some interesting habits uh how many dozens of cups of coffees do you drink a day what's the computer um that 

In [35]:
parser = StrOutputParser()

In [36]:
main_chain = parallel_chain | prompt | llm | parser

In [37]:
main_chain.invoke('Can you summarize the video')

'Based on the provided transcript, the conversation covers the following key points:\n\n* **Understanding the Universe and Biology:** The speakers discuss how fundamental physics and basic biological building blocks can help us understand complex systems, the universe, and the human mind, noting that humans are building computers to help the universe understand itself.\n* **AlphaFold and Protein Folding:** They highlight the protein folding problem, previously thought to be impossible to solve, which was solved by AlphaFold (specifically AlphaFold 2). This is described as one of the biggest breakthroughs in science and structural biology.\n* **The Importance of Proteins:** Proteins are described as essential to all life, functioning as "bio-nano machines" or the "workhorses of biology" that perform computations quickly to fold.\n* **Mapping the Human Proteome:** AlphaFold mapped the entire human proteome (all 20,000 proteins in the human body), revolutionizing structural biology by all

## Chat with the video

A simple loop on top of `main_chain` so you can ask as many questions as you like. Type `exit` to stop.

In [38]:
while True:
    user_question = input("You: ").strip()
    if user_question.lower() in {"exit", "quit", "q", ""}:
        print("Bye!")
        break
    print("Bot:", main_chain.invoke(user_question), "\n")

You: what is ai
Bot: I don't know. 

You: what is deep learning
Bot: I don't know. 

You: summarize the video
Bot: Based on the provided transcript, the conversation covers the following topics:

* **Physics and the Universe:** Discussing how deeper, more fundamental, or simpler explanations beyond the current standard model of physics could help us understand the world, and reflecting on how the human mind and computers are mechanisms through which the universe seeks to understand itself.
* **Biology and Complex Systems:** Transitioning to biology's basic building blocks to model complex systems, with the ultimate goal of understanding human biology and the mind.
* **AlphaFold and Protein Folding:** Highlighting AlphaFold (specifically AlphaFold 2) solving the protein folding problem, which was once considered impossible. 
* **Impact on Science:** Explaining how AlphaFold mapped the entire human proteome (all 20,000 proteins), revolutionizing structural biology by allowing scientists 

## Possible improvements (optional)

This is a basic RAG system. Industry-grade systems usually add:

- **UI:** a Streamlit app (paste a video URL and chat) or a Chrome extension that opens a chat panel on the YouTube page.
- **Evaluation:** measure faithfulness, answer relevancy, context precision and context recall with **RAGAS**; trace each pipeline step with **LangSmith**.
- **Indexing:** clean up auto-generated transcript errors, translate non-English transcripts into English, use a semantic chunker instead of fixed-size splitting, and use a managed vector DB (e.g. Pinecone) instead of FAISS.
- **Pre-retrieval:** query rewriting, multi-query generation, domain-aware routing.
- **Retrieval:** MMR search, hybrid (semantic + keyword) retrieval, re-ranking.
- **Post-retrieval:** contextual compression to keep only the useful parts of each chunk.
- **Augmentation:** better prompt templates and few-shot examples, answer grounding, context-window optimization.
- **Generation:** answer with citations, guardrails.
- **System design:** multimodal RAG, agentic RAG, memory-based (personalized) RAG.